### Instruction Finetuning

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os
import socket

In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "datasets" 

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [ ]:
# from importlib.metadata import version
# pkgs = [
#     "numpy",
#     "matplotlib",
#     "tiktoken",
#     "torch",
#     "tqdm",
#     # "tensorflow"
# ]
# for p in pkgs:
#     print(f"{p} version: {version(p)}")

numpy version: 2.5.2
matplotlib version: 3.11.2
tiktoken version: 0.14.0
torch version: 2.14.0+cu130
tqdm version: 4.70.1


In [8]:
import torch
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [9]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [10]:
if IS_PACE_SERVER:
    from model_training.post_training.artifacts.data.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
    from model_training.post_training.artifacts.data.instruction_tuning.prompt_style.styles import AlpacaFormat
    from model_training.post_training.artifacts.data.instruction_tuning.dataloader import InstructionDataset, custom_collate_fn
else:
    ## Fix this import
    raise Exception("Not Running on Pace")
        # from src.model import GPT

### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [11]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(ROOT_DATASET_DIR / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [12]:
alpaca = AlpacaFormat(data)

In [13]:
## Example of format
print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
print("=============================================")
print(alpaca.detailed_response(out_text = data[50]['output']))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

 #### Input:
Ocassion


### Response:
The correct spelling is 'Occasion.'


In [14]:
sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
print(sample)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


#### Spliting the data 

In [15]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [16]:
tokenizer = tiktoken.get_encoding("gpt2")
pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
pad_token_id

50256

In [17]:
dataloader = InstructionDataset(ds=data, tokenizer=tokenizer, prompt_style=alpaca)
(sample)

"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is an antonym of 'complicated'?\n\n### Response:\nAn antonym of 'complicated' is 'simple'."

#### Sample Data Prepare

In [18]:

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
custom_collate_fn(batch, pad_token_id=pad_token_id)

(tensor([[    0,     1,     2,     3,     4],
         [    5,     6, 50256, 50256, 50256],
         [    7,     8,     9, 50256, 50256]]),
 tensor([[    1,     2,     3,     4, 50256],
         [    6, 50256,  -100,  -100,  -100],
         [    8,     9, 50256,  -100,  -100]]))

In [19]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
tensor(0.7936)
tensor(1.1269)
loss_1 == loss_3: tensor(True)


### Creating, Training and Testing Dataset

In [20]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [26]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer, prompt_style=alpaca)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_dataset = InstructionDataset(val_data, tokenizer, prompt_style=alpaca)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer, prompt_style=alpaca)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [27]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 61]) torch.Size([8, 61])


In [30]:
from model_training.post_training.artifacts.model.hf_model.gpt2 import return_model

In [31]:
MODEL_NAME = 'gpt2'
model = return_model(MODEL_NAME)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
model.eval()

text_1 = "Every effort moves you"

NameError: name 'generate_text_simple' is not defined

In [33]:
# model = GPT.from_pretrained(config, "gpt2")
# print(f"Model Loaded: {model}")
# model.eval()
# model = model.to(device)